# Verify transcript findings on cleaned Silver data

Fresh organizer S3 inputs are ingested with the pipeline merged from main. This readout makes no network or model calls. The audit joins at transcript grain through interaction_id, validates unique keys, and compares exact text hashes to the saved Jev inputs. Scope: eight relevant tables, not full thirteen-table readiness.

Original labels, transcript-derived labels and human judgments remain distinct. Disagreement is not a verified error rate; structural cleaning does not resolve semantic correctness.

In [ ]:
import json, os
from pathlib import Path
root = Path(os.environ.get("ARABICA_REPO", Path.cwd())).resolve()
if not (root / "data_foundation").exists(): root = root.parent
run = root / "data_foundation/runs/silver-transcript-verification"
s = json.loads((run / "summary.json").read_text())
print("Snapshot:", s["generated_at_utc"])
print("Code commit:", s["code_commit"])
print("Quality warnings:", s["quality_warnings"])
print("Scope:", ", ".join(s["scope"]))

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
for key in ("population", "historical_comparison", "original_sample", "category_fields", "text_groups"):
    display(Markdown("### " + key.replace("_", " ").title()))
    display(pd.DataFrame([s[key]]))
display(pd.DataFrame(s["bronze_silver_changes"]))

In [ ]:
comparison = pd.DataFrame(s["labels_vs_saved_jev"])
display(comparison)
covered = comparison.loc[comparison.intent.notna(), "interactions"].sum()
mapped_disagreements = comparison.loc[(comparison.intent == "balance_inquiry") & (comparison.contact_reason != "Transaccional"), "interactions"].sum()
print(f"Exact-text prediction coverage: {covered:,}/{s['population']['transcripts']:,}")
print(f"Balance → Transaccional mapping disagreements: {mapped_disagreements:,}; not adjudicated errors.")
display(pd.DataFrame(s["complaints"]))
print(s["limits"])